# 04 - Reproduce the Stage 4 Threshold-Sensitivity Table

## Purpose

This notebook reproduces manuscript Table A6, which compares Stage 4 ESCO occupation-resolution accuracy at cosine-similarity thresholds from 0.5 to 0.9. Accuracy is recalculated from archived aggregate counts of compared and correct classifications.

## Input and output

- Input: `data/data-pipeline-answers/threshold-selection/threshold_accuracy_counts.csv`
- Output: `output/data-pipeline-answers/threshold-selection/classification_accuracy_by_threshold.csv`

Run from `notebooks/data-pipeline-answers/`.

## Historical-data limitation

The original Jooble vacancy records cannot be published. In addition, the historical vacancy-level predictions for threshold 0.7 were overwritten after the archived sensitivity table was produced. The included analysis-ready file therefore preserves the non-disclosive aggregate counts underlying the archived percentages. Counts for thresholds 0.5, 0.6, 0.8, and 0.9 were verified against the retained computed threshold datasets. For threshold 0.7, the integer counts were recovered from the archived two-decimal percentages and the common denominator of 137 non-missing prediction/manual-code pairs.

This workflow reproduces the published table but does not rerun the historical vacancy-level Stage 4 classifications.

In [ ]:
from pathlib import Path

import pandas as pd

# Explicit repository-relative paths are intentional for reviewer-answer analyses.
INPUT_FILE = Path(
    "../../data/data-pipeline-answers/threshold-selection/"
    "threshold_accuracy_counts.csv"
)
OUTPUT_DIR = Path(
    "../../output/data-pipeline-answers/threshold-selection"
)
OUTPUT_FILE = OUTPUT_DIR / "classification_accuracy_by_threshold.csv"

## 1. Load and validate the archived aggregate counts

The checks below prevent missing threshold/level combinations, duplicated rows, impossible counts, or silent changes to the archived two-decimal results.

In [ ]:
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Threshold count file is missing: {INPUT_FILE}")

counts = pd.read_csv(INPUT_FILE, sep=";", dtype={"threshold": "string"})
required_columns = {
    "threshold",
    "esco_level",
    "compared",
    "correct",
    "archived_accuracy_pct",
}
missing_columns = required_columns - set(counts.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

threshold_order = ["0.5", "0.6", "0.7", "0.8", "0.9"]
level_order = [
    "Major group",
    "Sub-major group",
    "Minor group",
    "Unit group",
    "ESCO sub-unit",
    "Full code",
]
expected_pairs = {
    (threshold, level)
    for threshold in threshold_order
    for level in level_order
}
observed_pairs = set(zip(counts["threshold"], counts["esco_level"]))
if observed_pairs != expected_pairs:
    raise ValueError(
        "Threshold/ESCO-level combinations differ from the expected 5 x 6 grid."
    )
if counts.duplicated(["threshold", "esco_level"]).any():
    raise ValueError("Duplicate threshold/ESCO-level rows found.")
if (counts["compared"] <= 0).any():
    raise ValueError("Every row must contain at least one compared record.")
if ((counts["correct"] < 0) | (counts["correct"] > counts["compared"])).any():
    raise ValueError("Correct counts must be between zero and compared counts.")

counts["accuracy_pct"] = (
    counts["correct"] / counts["compared"] * 100
).round(2)
if not counts["accuracy_pct"].equals(counts["archived_accuracy_pct"]):
    differences = counts.loc[
        counts["accuracy_pct"] != counts["archived_accuracy_pct"]
    ]
    raise ValueError(
        "Recalculated accuracy differs from the archived result:\n"
        f"{differences}"
    )

print(f"Validated aggregate rows: {len(counts)}")
print(counts.groupby("threshold")["compared"].first())

## 2. Construct manuscript Table A6

The manuscript reports one decimal place. Rows and columns are explicitly ordered to match the published table.

In [ ]:
table_a6 = counts.pivot(
    index="esco_level", columns="threshold", values="accuracy_pct"
).reindex(index=level_order, columns=threshold_order)
table_a6 = table_a6.round(1).rename_axis(
    index="ESCO level", columns=None
)
table_a6.columns = [f"th={threshold}" for threshold in table_a6.columns]
table_a6

## 3. Verify the published values and save the output

The assertion is an explicit manuscript consistency check. The output is generated from the archived integer counts, not copied from this expected-value table.

In [ ]:
expected_table_a6 = pd.DataFrame(
    [
        [78.2, 87.3, 87.6, 88.2, 88.8],
        [71.3, 84.5, 86.9, 87.4, 87.9],
        [56.4, 71.1, 73.0, 72.3, 72.4],
        [44.2, 57.0, 58.4, 55.5, 55.2],
        [37.2, 48.6, 50.4, 50.4, 50.0],
        [33.0, 44.4, 46.0, 45.4, 44.8],
    ],
    index=level_order,
    columns=[f"th={threshold}" for threshold in threshold_order],
).rename_axis(index="ESCO level")
pd.testing.assert_frame_equal(table_a6, expected_table_a6)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
table_a6.reset_index().to_csv(OUTPUT_FILE, sep=";", index=False)
print(f"Saved: {OUTPUT_FILE}")
table_a6